# M1: Wav2Lip — Talking Head Generation

## Overview

This notebook generates talking head animations from static headshots using **Wav2Lip**, a state-of-the-art lip-sync model.

### How it works

Wav2Lip requires a **video** input (not a static image). The pipeline therefore has two stages:

1. **Image → looped video**: Each static headshot PNG is converted into a short looped MP4 using FFmpeg (`-loop 1`). This gives Wav2Lip a valid video stream to process.
2. **Wav2Lip inference**: Wav2Lip synthesises lip movements driven by a 5-second silent WAV file. Because the audio is silence, the mouth will remain largely closed — this stage is used to verify the pipeline and model are functioning correctly. Swap in real speech audio during M2 for full animation.

### Wav2Lip strengths and trade-offs

| Aspect | Wav2Lip |
|---|---|
| **Lip sync accuracy** | Excellent — industry-leading |
| **Face region outside mouth** | May appear softer / blurrier (known trade-off) |
| **Face identity preservation** | Good overall |
| **Head motion** | None — input video is a static loop |

> **Note:** Wav2Lip pastes a synthesised mouth region back onto the face. The area immediately around the mouth can look blurry or have a visible boundary — this is expected behaviour and a documented limitation of the model. Compare with LivePortrait and SadTalker outputs (also generated in M1) before selecting a model for M2.

### Inputs / Outputs

- **Headshots:** `MyDrive/talking_head/headshots/person_01.png` … `person_08.png`
- **Outputs:** `MyDrive/talking_head/outputs/m1/wav2lip/person_01_talking.mp4` … `person_08_talking.mp4`
- **Runtime:** Google Colab with GPU (T4 free tier recommended)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 — Install dependencies, clone Wav2Lip, download checkpoint,
#           and generate the silent driver WAV
# ─────────────────────────────────────────────────────────────────────────────

import os, subprocess, sys

# ── 1. Clone Wav2Lip ──────────────────────────────────────────────────────────
if not os.path.exists('/content/Wav2Lip'):
    print('Cloning Wav2Lip ...')
    subprocess.run(
        ['git', 'clone', 'https://github.com/Rudrabha/Wav2Lip', '/content/Wav2Lip'],
        check=True
    )
    print('Wav2Lip cloned.')
else:
    print('Wav2Lip already cloned, skipping.')

# ── 2. Install Python dependencies ───────────────────────────────────────────
print('Installing Python dependencies ...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'librosa==0.9.2',
     'numba==0.56.4',
     'resampy',
     'face_alignment',
     'batch_face_detection',
     'opencv-python-headless',
     'tqdm'],
    check=True
)

# Install Wav2Lip's own requirements (some may already be satisfied)
req_file = '/content/Wav2Lip/requirements.txt'
if os.path.exists(req_file):
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '-r', req_file],
        check=True
    )
print('Dependencies installed.')

# ── 3. Download pretrained Wav2Lip GAN checkpoint via gdown ──────────────────
CHECKPOINT_DIR = '/content/Wav2Lip/checkpoints'
CHECKPOINT_PATH = f'{CHECKPOINT_DIR}/wav2lip_gan.pth'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if not os.path.exists(CHECKPOINT_PATH):
    print('Downloading wav2lip_gan.pth ...')
    # Official Wav2Lip GAN checkpoint (Google Drive file ID)
    GDRIVE_FILE_ID = '1cyFcRxlB8bYJCFHmJ7Yv4sECqXdkBV3c'
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'gdown'],
        check=True
    )
    subprocess.run(
        ['python', '-m', 'gdown', f'https://drive.google.com/uc?id={GDRIVE_FILE_ID}',
         '-O', CHECKPOINT_PATH],
        check=True
    )
    if os.path.exists(CHECKPOINT_PATH):
        size_mb = os.path.getsize(CHECKPOINT_PATH) / (1024 * 1024)
        print(f'Checkpoint downloaded: {size_mb:.1f} MB')
    else:
        raise RuntimeError(
            'Checkpoint download failed. '
            'Visit https://github.com/Rudrabha/Wav2Lip#getting-the-weights '
            'and place wav2lip_gan.pth in /content/Wav2Lip/checkpoints/ manually.'
        )
else:
    print('Checkpoint already present, skipping download.')

# ── 4. Generate 5-second silent driver WAV ───────────────────────────────────
import numpy as np
import scipy.io.wavfile as wav

DRIVER_AUDIO = '/content/silent_driver.wav'
sample_rate = 16000
silence = np.zeros(sample_rate * 5, dtype=np.int16)
wav.write(DRIVER_AUDIO, sample_rate, silence)
print(f'Silent driver WAV created: {DRIVER_AUDIO}')

# ── 5. Confirm GPU is available ───────────────────────────────────────────────
import subprocess as _sp
gpu_info = _sp.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                   capture_output=True, text=True)
if gpu_info.returncode == 0:
    print(f'GPU detected: {gpu_info.stdout.strip()}')
else:
    print('WARNING: No GPU detected. Enable GPU in Runtime > Change runtime type > T4 GPU.')

print('\nSetup complete')

In [ ]:
from pathlib import Path

# Detect whether we are running in Google Colab or locally in Jupyter
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    DRIVE_BASE    = '/content/drive/MyDrive/talking_head'
    HEADSHOTS_DIR = f'{DRIVE_BASE}/headshots'
    OUTPUT_DIR    = f'{DRIVE_BASE}/outputs/m1/wav2lip'
    DRIVER_AUDIO = '/content/silent_driver.wav'
    CHECKPOINT_PATH = '/content/Wav2Lip/checkpoints/wav2lip_gan.pth'
    print("Running in Google Colab")
except ImportError:
    import sys
    IN_COLAB = False
    PROJECT_ROOT  = Path().resolve()
    sys.path.insert(0, str(PROJECT_ROOT))
    HEADSHOTS_DIR = str(PROJECT_ROOT / 'headshots')
    OUTPUT_DIR    = str(PROJECT_ROOT / 'outputs' / 'm1' / 'wav2lip')
    DRIVER_AUDIO = ''  # Leave blank — silent driver is auto-generated
    print(f"Running locally. Project root: {PROJECT_ROOT}")

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Headshots : {HEADSHOTS_DIR}")
print(f"Output dir: {OUTPUT_DIR}")


In [ ]:
import glob, os
from pathlib import Path

if IN_COLAB:
    # ---- Inline definition (Colab only) ------------------------------------
    # When running in Colab the local scripts aren't available, so the function
    # is defined here. When running locally this block is skipped entirely and
    # the function is imported from scripts/m1/wav2lip.py instead.
    import subprocess, shutil

    def generate_animation(headshot_path, output_dir, **kwargs):
        raise NotImplementedError(
            "Replace this stub with the actual Colab inference code from the notebook "
            "you ran previously, or use the full Colab notebook as-is."
        )
else:
    # ---- Local import (no code duplication) --------------------------------
    from scripts.m1.wav2lip import generate_animation
    print("generate_animation imported from scripts/m1/wav2lip.py")

# ── Batch: run all headshots ─────────────────────────────────────────────────
headshots = sorted(glob.glob(os.path.join(HEADSHOTS_DIR, "person_*.png")))
print(f"Found {len(headshots)} headshot(s)")

results, failures = [], []
for hs in headshots:
    name = Path(hs).stem
    print(f"Processing {name} ...", end=" ", flush=True)
    try:
        r = generate_animation(hs, OUTPUT_DIR, driver_audio=DRIVER_AUDIO)
        results.append(r)
        size_kb = os.path.getsize(r["output_path"]) // 1024
        print(f"Done ({size_kb} KB)")
    except Exception as e:
        failures.append({"headshot": hs, "error": str(e)})
        print(f"FAILED: {e}")

print(f"
Completed {len(results)}/{len(headshots)}  |  Failed: {len(failures)}")


# Quality Check — Wav2Lip Outputs

Review each of the 8 output videos saved to `MyDrive/talking_head/outputs/m1/wav2lip/` against the checklist below.

## Per-video checklist

- [ ] Lip / mouth movement looks natural and matches the audio timing
- [ ] Face region outside the mouth: some blurriness is expected with Wav2Lip — check it is acceptable
- [ ] No severe flickering or face-box artifacts around the mouth area
- [ ] Overall face identity preserved (person still recognisable)
- [ ] Loop point is seamless (no hard cut at the 5-second boundary)

## Expected output files

| File | Status |
|---|---|
| `person_01_talking.mp4` | ☐ Reviewed |
| `person_02_talking.mp4` | ☐ Reviewed |
| `person_03_talking.mp4` | ☐ Reviewed |
| `person_04_talking.mp4` | ☐ Reviewed |
| `person_05_talking.mp4` | ☐ Reviewed |
| `person_06_talking.mp4` | ☐ Reviewed |
| `person_07_talking.mp4` | ☐ Reviewed |
| `person_08_talking.mp4` | ☐ Reviewed |

## Comparison note

**Wav2Lip** has the best lip-sync accuracy of the three M1 models, but the face region outside the mouth may look softer or blurrier than **LivePortrait** or **SadTalker**. This is a known trade-off: Wav2Lip pastes a synthesised mouth crop back onto the original face, which can leave a visible seam or softness around the mouth boundary.

Before selecting a model for M2, compare all three outputs side-by-side on the same person:

| Model | Lip sync | Face naturalness | Head motion |
|---|---|---|---|
| **Wav2Lip** | Excellent | Moderate (mouth area blurry) | None |
| LivePortrait | Good | Better overall | Subtle |
| SadTalker | Good | Good | Yes (3D head pose) |

Use the model that best balances lip-sync quality and visual naturalness for the client's use case.